# Progress Report Cross-Score Comparison

This notebook aggregates the independent progress CSVs on the shared 13-pair practice subset (878 buildings), computes pairwise Spearman rank correlations to verify score complementarity, and generates summary comparison figures for the IEEE progress report.

In [10]:
import csv
import json
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

ROOT = Path('..').resolve()
DATASET_LABELS = ROOT.parent / 'Dataset' / 'train' / 'labels'
OUT_FIG_DIR = ROOT.parent / 'progress' / 'figures'
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

SCORES = [
    ('Correlation Loss', ROOT / 'outputs' / 'correlation_loss' / 'progress' / 'tables' / 'correlation_subset.csv', ROOT / 'outputs' / 'correlation_loss' / 'progress' / 'tables' / 'correlation_metrics.txt'),
    ('Change-Vector Mag.', ROOT / 'outputs' / 'change_vector' / 'progress' / 'tables' / 'change_vector_subset.csv', ROOT / 'outputs' / 'change_vector' / 'progress' / 'tables' / 'change_vector_metrics.txt'),
    ('Morphological Prof.', ROOT / 'outputs' / 'morphology' / 'progress' / 'tables' / 'morphology_subset.csv', ROOT / 'outputs' / 'morphology' / 'progress' / 'tables' / 'morphology_metrics.txt'),
    ('Texture Contrast', ROOT / 'outputs' / 'texture_contrast' / 'progress' / 'tables' / 'texture_subset.csv', ROOT / 'outputs' / 'texture_contrast' / 'progress' / 'tables' / 'texture_metrics.txt'),
    ('Lab Histogram Dist.', ROOT / 'outputs' / 'lab_histogram' / 'progress' / 'tables' / 'lab_histogram_subset.csv', ROOT / 'outputs' / 'lab_histogram' / 'progress' / 'tables' / 'lab_histogram_metrics.txt'),
]

active_scores = [(name, csv_p, met_p) for name, csv_p, met_p in SCORES if csv_p.exists()]
print('Loaded scores:', [s[0] for s in active_scores])

Loaded scores: ['Correlation Loss', 'Change-Vector Mag.', 'Morphological Prof.', 'Texture Contrast', 'Lab Histogram Dist.']


In [11]:
# Load CSVs and align by (image_id, building_id)
score_maps = {}
keys_order = None
for name, csv_p, _ in active_scores:
    with open(csv_p, newline='') as f:
        rows = list(csv.DictReader(f))
    if keys_order is None:
        keys_order = [(r['image_id'], r['building_id']) for r in rows]
    score_maps[name] = {(r['image_id'], r['building_id']): float(r['score']) for r in rows}

# Load ground-truth damage labels
label_map = {}
for img_id in sorted(set(k[0] for k in keys_order)):
    lbl_file = DATASET_LABELS / f'{img_id}_post_disaster.json'
    with open(lbl_file) as f:
        data = json.load(f)
    for feat in data.get('features', {}).get('xy', []):
        uid = feat['properties']['uid']
        subtype = feat['properties'].get('subtype', 'un-classified')
        label_map[(img_id, uid)] = subtype

labels = [label_map[k] for k in keys_order]
events = [k[0].rsplit('_', 1)[0] for k in keys_order]
print('Total aligned buildings:', len(keys_order))
for cls in ['no-damage', 'minor-damage', 'major-damage', 'destroyed']:
    print(f'  {cls}: {sum(1 for l in labels if l == cls)}')

Total aligned buildings: 878
  no-damage: 687
  minor-damage: 8
  major-damage: 14
  destroyed: 169


In [12]:
# Parse metrics files for F1 and FPR
metrics_summary = {}
for name, _, met_p in active_scores:
    if not met_p.exists():
        continue
    txt = met_p.read_text()
    info = {}
    for line in txt.splitlines():
        if line.startswith('threshold,'):
            parts = [p.strip() for p in line.split(',')]
            for i in range(0, len(parts), 2):
                info[parts[i]] = float(parts[i+1])
        elif line.startswith('undamaged_false_positive_rate,'):
            parts = [p.strip() for p in line.split(',')]
            for i in range(0, len(parts), 2):
                info[parts[i]] = float(parts[i+1])
    metrics_summary[name] = info
    print(f"{name:22s} | F1={info['destroyed_f1']:.4f} | FPR={info['undamaged_false_positive_rate']:.4f} | P={info['precision']:.4f} | R={info['recall']:.4f} | cut={info['threshold']:.3f}")

Correlation Loss       | F1=0.4920 | FPR=0.2285 | P=0.4000 | R=0.6391 | cut=0.770
Change-Vector Mag.     | F1=0.6143 | FPR=0.1557 | P=0.5139 | R=0.7633 | cut=50.160
Morphological Prof.    | F1=0.6319 | FPR=0.1048 | P=0.5897 | R=0.6805 | cut=7.056
Texture Contrast       | F1=0.5658 | FPR=0.1630 | P=0.4872 | R=0.6746 | cut=0.210
Lab Histogram Dist.    | F1=0.5956 | FPR=0.0699 | P=0.6333 | R=0.5621 | cut=0.655


In [13]:
# Compute pairwise Spearman rank correlation across the 878 buildings
def rankdata_simple(a):
    order = np.argsort(a, kind='mergesort')
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(a), dtype=float)
    # average ties
    sorted_a = a[order]
    i = 0
    while i < len(sorted_a):
        j = i + 1
        while j < len(sorted_a) and sorted_a[j] == sorted_a[i]:
            j += 1
        if j - i > 1:
            avg = np.mean(ranks[order[i:j]])
            ranks[order[i:j]] = avg
        i = j
    return ranks

names = [s[0] for s in active_scores]
mat = np.array([[score_maps[n][k] for k in keys_order] for n in names])
ranked = np.array([rankdata_simple(row) for row in mat])
corr = np.corrcoef(ranked)

print('Pairwise Spearman correlation matrix:')
header = '                      ' + '  '.join(f'{n[:10]:>10s}' for n in names)
print(header)
for i, n in enumerate(names):
    row_str = '  '.join(f'{corr[i, j]:10.4f}' for j in range(len(names)))
    print(f'{n:22s}{row_str}')

Pairwise Spearman correlation matrix:
                      Correlatio  Change-Vec  Morphologi  Texture Co  Lab Histog
Correlation Loss          1.0000      0.3814      0.3759      0.2119      0.3275
Change-Vector Mag.        0.3814      1.0000      0.7146      0.4959      0.3544
Morphological Prof.       0.3759      0.7146      1.0000      0.5193      0.2833
Texture Contrast          0.2119      0.4959      0.5193      1.0000      0.2700
Lab Histogram Dist.       0.3275      0.3544      0.2833      0.2700      1.0000


In [ ]:
# Figure 1: F1 vs Undamaged FPR operating points
plt.rcParams.update({'font.size': 10, 'figure.dpi': 200})
fig, ax = plt.subplots(figsize=(6.2, 4.0))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
for idx, n in enumerate(names):
    if n not in metrics_summary:
        continue
    f1 = metrics_summary[n]['destroyed_f1']
    fpr = metrics_summary[n]['undamaged_false_positive_rate']
    ax.scatter(fpr, f1, s=110, color=colors[idx % len(colors)], zorder=5, edgecolors='black', linewidth=0.8)
    dx, dy = 0.006, 0.008
    offsets = {'Correlation': (-0.055, 0.014), 'Lab': (-0.012, -0.040), 'Texture': (0.008, -0.012)}
    for key, (ox, oy) in offsets.items():
        if key in n:
            dx, dy = ox, oy
    ax.annotate(f'{n}\n({fpr:.3f}, {f1:.3f})', (fpr, f1), xytext=(fpr + dx, f1 + dy), fontsize=8.5)

ax.axvline(0.25, color='gray', linestyle='--', linewidth=1.1, label='FPR Minimum Cap (0.25)')
ax.axvline(0.10, color='green', linestyle=':', linewidth=1.2, label='FPR Target (0.10)')
ax.axhline(0.35, color='gray', linestyle='-.', linewidth=1.1, label='F1 Minimum (0.35)')
ax.axhline(0.60, color='green', linestyle=':', linewidth=1.2, label='F1 Target (0.60)')

ax.set_xlim(0.0, 0.28)
ax.set_ylim(0.30, 0.72)
ax.set_xlabel('Undamaged False-Positive Rate (lower is better)')
ax.set_ylabel('Destroyed-vs-Rest F1 (higher is better)')
ax.set_title('Individual Score Operating Points (13-Pair Practice Subset)')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower left', fontsize=8, framealpha=0.9)
fig.tight_layout()
fig.savefig(OUT_FIG_DIR / 'f1_vs_fpr_comparison.png', dpi=220)
plt.close(fig)
print('Saved f1_vs_fpr_comparison.png')

Saved f1_vs_fpr_comparison.png


In [15]:
# Figure 2: Pairwise Spearman correlation heatmap
fig, ax = plt.subplots(figsize=(5.4, 4.4))
im = ax.imshow(corr, vmin=0.0, vmax=1.0, cmap='YlGnBu')
ax.set_xticks(range(len(names)))
ax.set_yticks(range(len(names)))
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8.5)
ax.set_yticklabels(names, fontsize=8.5)
for i in range(len(names)):
    for j in range(len(names)):
        val = corr[i, j]
        txt_color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=txt_color, fontsize=9.5, fontweight='bold')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Spearman Rank Correlation')
ax.set_title('Inter-Score Rank Correlation (n = 878 Buildings)')
fig.tight_layout()
fig.savefig(OUT_FIG_DIR / 'score_correlation_heatmap.png', dpi=220)
plt.close(fig)
print('Saved score_correlation_heatmap.png')

Saved score_correlation_heatmap.png


In [16]:
# Figure 3: Hazard-by-hazard Destroyed / Undamaged separation ratio
event_list = ['santa-rosa-wildfire', 'socal-fire', 'midwest-flooding', 'guatemala-volcano']
event_labels = ['Santa Rosa\nWildfire', 'SoCal\nFire', 'Midwest\nFlooding', 'Guatemala\nVolcano']
fig, ax = plt.subplots(figsize=(6.8, 3.8))
x = np.arange(len(event_list))
w = 0.8 / max(1, len(names))
for idx, n in enumerate(names):
    ratios = []
    for ev in event_list:
        nd_vals = [score_maps[n][k] for k, l, e in zip(keys_order, labels, events) if e == ev and l == 'no-damage']
        ds_vals = [score_maps[n][k] for k, l, e in zip(keys_order, labels, events) if e == ev and l == 'destroyed']
        nd_m = np.mean(nd_vals) if nd_vals else 1e-9
        ds_m = np.mean(ds_vals) if ds_vals else 0.0
        ratios.append(ds_m / max(nd_m, 1e-9))
    offset = (idx - (len(names) - 1) / 2.0) * w
    ax.bar(x + offset, ratios, width=w * 0.92, label=n, color=colors[idx % len(colors)], edgecolor='black', linewidth=0.5)
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.0, label='No Separation (Ratio = 1.0)')
ax.set_xticks(x)
ax.set_xticklabels(event_labels, fontsize=8.5)
ax.set_ylabel('Destroyed Mean / Undamaged Mean')
ax.set_title('Hazard-Wise Class Separation Ratio (Higher = Stronger Signal)')
ax.set_ylim(0, 4.0)
ax.grid(True, axis='y', alpha=0.3)
ax.legend(fontsize=7.5, ncol=2, loc='upper left')
fig.tight_layout()
fig.savefig(OUT_FIG_DIR / 'hazard_profile_bars.png', dpi=220, bbox_inches='tight')
plt.close(fig)
print('Saved hazard_profile_bars.png')

Saved hazard_profile_bars.png
